# PhySim02 — PD Control and Robot DOFs

### Lab Description

A robot arm collapses under gravity when no controller supports it. This lab introduces Genesis joint-level control and shows how proportional–derivative (PD) gains produce stable, physically consistent motion.

Using the Franka Emika Panda, you will inspect its seven arm joints and two gripper joints, then compare direct state setting with position, velocity, and force control.

#### Recommended Hardware

An AMD GPU supported by ROCm, such as an AMD Radeon™ GPU or AMD Ryzen™ AI processor with integrated Radeon graphics.

#### Software Environment

OS: Ubuntu 24.04 LTS  
Install [AUP Learning Cloud](https://amdresearch.github.io/aup-learning-cloud/installation/quick-start.html). The Genesis Simulation image provides ROCm, PyTorch, and `genesis-world==1.3.1`.

## Goals

- Identify the Franka Panda joints and nine degrees of freedom.
- Configure proportional, derivative, and force-limit parameters.
- Compare direct state updates with physically consistent control commands.
- Apply position, velocity, and force control to selected DOFs.
- Record and compare the resulting robot motion.

In [ ]:
# Suppress warning messages for clearer output
import os
import warnings

os.environ["TI_LOG_LEVEL"] = "error"
warnings.filterwarnings("ignore")
os.makedirs("Videos", exist_ok=True)

## Init and Create a Scene

In PhySim01, we used mostly default settings when creating the scene. Here we use more advanced simulator and viewer options.

You can customize the **Simulator** and **Visualizer** through detailed configuration options when creating a scene.

**Simulator options** define the physical simulation behavior. Common parameters include:

* **dt** – Duration of each simulation step (in seconds)
* **gravity** – Gravity force vector (N/kg)
* **floor_height** – Ground plane height

**Visualizer options** control the virtual camera and rendering behavior. Key parameters:

* **camera_pos** – Initial position of the camera
* **camera_lookat** – Target point the camera focuses on
* **camera_fov** – Field of view (in degrees)
* **max_FPS** – Set max FPS.


In [ ]:
import genesis as gs
import numpy as np

assert "scene" not in globals(), "Scene already exists. Restart the kernel before rerunning this lab."

########################## init ##########################
gs.init(backend=gs.amdgpu, theme="light")

########################## create a scene ##########################
scene = gs.Scene(
    viewer_options=gs.options.ViewerOptions(
        camera_pos=(0, -3.5, 2.5),
        camera_lookat=(0.0, 0.0, 0.5),
        camera_fov=30,
        max_FPS=60,
    ),
    sim_options=gs.options.SimOptions(
        dt=0.01,
    ),
    show_viewer=False,
)

## Add Entities and Build the Scene

Just like what we did in Lab 1, we add a **plane**, an **arm**, and a **camera** to the scene, and then build the scene.

In [ ]:
########################## entities ##########################
plane = scene.add_entity(
    gs.morphs.Plane(),
)
franka = scene.add_entity(
    gs.morphs.MJCF(
        file="xml/franka_emika_panda/panda.xml",
    ),
)
cam = scene.add_camera(
    res=(640, 480),
    pos=(3.5, 0.0, 2.5),
    lookat=(0, 0, 0.5),
    fov=30,
    GUI=True,
)

########################## build ##########################
scene.build()
print("Successfully built the scene.")

## Control Joints and DOFs

In robotics, the terms **joint** and **degree of freedom (DOF)** are related but not quite the same. 

A joint is the physical connection between two parts (or links) of a robot that allows relative motion. For example, a hinge, a slider, or a ball-and-socket connection. Each joint enables certain types of movement.

A degree of freedom (DOF), on the other hand, refers to the number of independent ways a joint (or the entire robot) can move. For instance, a revolute (rotational) joint has one DOF because it can rotate around a single axis, while a spherical joint has three DOFs, it can rotate around three perpendicular axes.

![image.png](attachment:267b2468-7d14-45fd-a20d-3eb514f2c857.png)

Take Franka Panda arm for example, it has 7 revolute joints in the arm and 2 prismatic joints in its gripper. Since each joint has only 1 DOF, the robot ends up with 9 DOFs in total.

In [ ]:
jnt_names = [
    "joint1",
    "joint2",
    "joint3",
    "joint4",
    "joint5",
    "joint6",
    "joint7",
    "finger_joint1",
    "finger_joint2",
]

dofs_idx_temp = [franka.get_joint(name).dofs_idx_local for name in jnt_names]
dofs_idx = [idx for sublist in dofs_idx_temp for idx in sublist]

print(dofs_idx)

## Control Gains

Control gains decide how much torque the controller applies to reduce errors in position or velocity. URDF/MJCF files usually provide default values, but manual tuning is often necessary for stable, realistic control.

Genesis exposes three functions:

* `.set_dofs_kp` — proportional gains
* `.set_dofs_kv` — derivative gains
* `.set_dofs_force_range` — safety limits on torque/force

Together, `kp` and `kv` form the **PD controller**:

For Franka, the arm joints (joint1–joint7) use higher gains for stiffness and precision, and the finger joints use lower gains so they feel softer and safer when grasping objects. 

A typical setup looks like this:

In [ ]:
############ Optional: set control gains ############

# set positional gains
franka.set_dofs_kp(
    kp=np.array([4500, 4500, 3500, 3500, 2000, 2000, 2000, 100, 100]),
    dofs_idx_local=dofs_idx,
)
# set velocity gains
franka.set_dofs_kv(
    kv=np.array([450, 450, 350, 350, 200, 200, 200, 10, 10]),
    dofs_idx_local=dofs_idx,
)
# set force range for safety
franka.set_dofs_force_range(
    lower=np.array([-87, -87, -87, -87, -12, -12, -12, -100, -100]),
    upper=np.array([87, 87, 87, 87, 12, 12, 12, 100, 100]),
    dofs_idx_local=dofs_idx,
)

## Directly Setting DOF Positions

It’s also possible to set DOF positions directly with `.set_dofs_position`. This instantly changes the robot state and bypasses physics. This can be useful for resets or demonstrations, but it may create unrealistic motion that violates physical laws.

To avoid overly lengthy log information, we will not print the detailed logs here, we’ll use a progress bar to indicate the progress.


In [ ]:
import logging

from tqdm import tqdm

# Set logger to warning to avoid log info.
gs.logger._logger.setLevel(logging.WARNING)

# Camera recording
rgb, depth, segmentation, normal = cam.render(rgb=True, depth=True, segmentation=True, normal=True)
cam.start_recording(save_to_filename="Videos/video_02.mp4", fps=60)

# Hard reset
for i in tqdm(range(150), ncols=100):
    if i < 50:
        franka.set_dofs_position(np.array([1, 1, 0, 0, 0, 0, 0, 0.04, 0.04]), dofs_idx)
    elif i < 100:
        franka.set_dofs_position(np.array([-1, 0.8, 1, -2, 1, 0.5, -0.5, 0.04, 0.04]), dofs_idx)
    else:
        franka.set_dofs_position(np.array([0, 0, 0, 0, 0, 0, 0, 0, 0]), dofs_idx)
    cam.render()
    scene.step()

## Using the PD Controller

To respect physics, use the `control_*` APIs instead. These send commands to the PD controller rather than overwriting the state. We can save it as a complete video and check the result.

In [ ]:
# PD control
for i in tqdm(range(1250), ncols=100):
    if i == 0:
        franka.control_dofs_position(
            np.array([1, 1, 0, 0, 0, 0, 0, 0.04, 0.04]),
            dofs_idx,
        )
    elif i == 250:
        franka.control_dofs_position(
            np.array([-1, 0.8, 1, -2, 1, 0.5, -0.5, 0.04, 0.04]),
            dofs_idx,
        )
    elif i == 500:
        franka.control_dofs_position(
            np.array([0, 0, 0, 0, 0, 0, 0, 0, 0]),
            dofs_idx,
        )
    elif i == 750:
        # control first dof with velocity, and the rest with position
        franka.control_dofs_position(
            np.array([0, 0, 0, 0, 0, 0, 0, 0, 0])[1:],
            dofs_idx[1:],
        )
        franka.control_dofs_velocity(
            np.array([1.0, 0, 0, 0, 0, 0, 0, 0, 0])[:1],
            dofs_idx[:1],
        )
    elif i == 1000:
        franka.control_dofs_force(
            np.array([0, 0, 0, 0, 0, 0, 0, 0, 0]),
            dofs_idx,
        )

    cam.render()
    scene.step()

cam.stop_recording()

## Show the video

In the video, you’ll first see three relatively rigid movements, these are discontinuous actions created using set_dofs_position. The following smoother, continuous motions are produced by the control_* APIs. Therefore, when creating robots in a virtual environment that behave according to physical laws, we usually use the control_* APIs to write the program.

In [ ]:
from IPython.display import Video

Video(url="Videos/video_02.mp4")

## Conclusions

You identified the Franka Panda DOFs, configured PD gains and force limits, and compared direct state updates with position, velocity, and force control. In PhySim03, these control primitives become a complete IK-guided grasping sequence.

---

Copyright (C) 2026 Advanced Micro Devices, Inc. All rights reserved. Portions of this file consist of AI-generated content.  
SPDX-License-Identifier: MIT